# 課題と解答例：20_pattern_foundations

元Notebook: [../20_pattern_foundations.ipynb](../20_pattern_foundations.ipynb)

## 課題

1. reaction_rateとdiffusionの両方を0にし，初期状態が変わらないことを確認する．
2. diffusionを0.10，0.20，0.40と変え，空間分散がどのように変化するか表にする．
3. reaction_rateとdiffusionの両方を正にし，反応だけ，拡散だけの場合と比較する．各項の効果が図のどこに現れたか説明する．
4. 初期状態を一様な配列へ変える．拡散だけでは値が変化しない理由を5点ラプラシアンから説明する．

### 追加実装課題

5. `summarize_field(u)` という関数を作り，最小値，最大値，平均値，空間分散を辞書で返す．
6. `run_reaction_diffusion_table(reaction_rates, diffusions)` を作り，複数条件の結果をDataFrameにまとめる．
7. `reaction_rate=0.4` かつ `diffusion=0.20` の結果を，反応だけ・拡散だけの結果と同じ色範囲で並べて描く関数を作る．

## 解答例

1. `reaction_rate=0.0`, `diffusion=0.0` なら更新式の右辺が0になるので，初期状態は変わらない．

   ```python
   u_same = evolve(u0, reaction_rate=0.0, diffusion=0.0)
   np.allclose(u_same, u0)
   ```

2. `diffusion` を大きくすると，凹凸がより速くならされるため，空間分散は小さくなる．

   ```python
   for d in [0.10, 0.20, 0.40]:
       u_d = evolve(u0, reaction_rate=0.0, diffusion=d)
       print(d, u_d.var())
   ```

3. 両方を正にすると，反応は各格子点の値を増やし，拡散は空間的な差をならす．反応だけでは平均値が変わり，拡散だけでは平均値をほぼ保ちながら分散が下がる．

4. 一様な配列では，どの格子点も上下左右と同じ値を持つため，5点ラプラシアンは0になる．拡散項が0なので，拡散だけでは値が変化しない．

5. 実装例である．

   ```python
   def summarize_field(u):
       return {
           "min": float(u.min()),
           "max": float(u.max()),
           "mean": float(u.mean()),
           "variance": float(u.var()),
       }
   ```

6. 条件表は次のように作れる．

   ```python
   def run_reaction_diffusion_table(reaction_rates, diffusions):
       rows = []
       for rr in reaction_rates:
           for d in diffusions:
               u = evolve(u0, reaction_rate=rr, diffusion=d)
               row = {"reaction_rate": rr, "diffusion": d}
               row.update(summarize_field(u))
               rows.append(row)
       return pd.DataFrame(rows)
   ```

7. 比較図は，同じ `vmin` と `vmax` を指定して描く．

   ```python
   def plot_comparison(fields, titles):
       fig, axes = plt.subplots(1, len(fields), figsize=(4*len(fields), 3.5))
       for ax, field, title in zip(axes, fields, titles):
           ax.imshow(field, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
           ax.set_title(title)
           ax.axis("off")
       return fig
   ```